# 04 · Choosing how to build the model

Every decision about the model is made here, using only data from before 1 September 2024, the day
the test period starts. Tuning on the same months I test on would make the model look better than
it really is.

The practice runs work like the final test. On the first of April, May, June and July the model
learns from what was known by then and is scored on the quotes sent that month. Quotes sent after
17 July are left out, because some of their replies hadn't arrived by 1 September.

The yardstick is **log loss**, a standard score for predicted chances that punishes confident
mistakes (lower is better). When two options tie on it, I pick the one whose chances hold up better
group by group. The tables also show **AUC**: how often a quote that said yes was rated higher than
one that said no (0.5 is a coin flip, 1 is perfect).

In [1]:
import sys

sys.path.insert(0, "../scripts")

import pandas as pd

from quote_followup.config import load_settings
from quote_followup.data import load_quotes

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 100)
settings = load_settings("../config.toml")
quotes = load_quotes(settings.data_dir)

import ablations

candidates = ablations.with_candidate_inputs(quotes)

## Which facts about a quote matter?

Every fact tried here is known the moment the quote goes out.

In [2]:
ablations.send_time_ablations(candidates, settings).round(4)

,option,rows,auc,log_loss
0,"type of work, customer source, sending speed (chosen)",5990,0.6388,0.6137
1,"+ industry, customer type, quote size, number of items",5990,0.6373,0.6153
2,+ the business's past win rate,5990,0.6386,0.6136
3,+ the customer's past win rate,5990,0.6355,0.6164
4,+ sent on a weekend,5990,0.6386,0.6136
5,without type of work,5990,0.5812,0.6319
6,without customer source,5990,0.6234,0.6185
7,without sending speed (quote size instead),5990,0.6295,0.6165
8,gradient boosting with every fact,5990,0.5867,0.7086


Three facts do the work: the type of work, how the customer found the business, and how fast the
quote went out. Industry, customer type, quote size, number of items, weekend sending and either
history rate add nothing, which matches notebook 02's finding that businesses don't differ beyond
their mix. Taking any one of the three out makes things worse, the type of work most of all.

A more complex model (gradient boosting) does clearly worse. With only a few real patterns to find,
its extra flexibility mostly picks up noise, and it couldn't explain its scores the way this one can.

## How best to use "days without a reply"?

Each option is scored at every point a quote was still waiting (after 0, 1, 2, 3, 5, 7, 10, 14 and
21 days). The last columns show how many groups of similar quotes each one misses by more than luck
explains.

In [3]:
ablations.silence_ablations(candidates, settings).round(4)

,option,rows,auc,log_loss,groups_missed_beyond_luck,worst_miss_z,luck_threshold_z
0,one says-no-or-goes-quiet figure per type of work,24151,0.7599,0.5717,3,4.6545,3.4101
1,says-no-or-goes-quiet chance per customer (chosen),24151,0.7616,0.5705,1,3.6614,3.4101
2,"same, reply speed also learned by customer source",24151,0.7612,0.5711,2,4.7092,3.4101
3,"same, reply speed also learned by sending speed",24151,0.7619,0.5704,3,4.7393,3.4101
4,"same, with gradient boosting instead",24151,0.7336,0.6137,7,4.8450,3.4101
5,days waiting as one more fact,24151,0.7592,0.5733,4,4.1777,3.4101
6,"days waiting as one more fact, gradient boosting",24151,0.7372,0.6257,3,3.8465,3.4101
7,days waiting only,24151,0.7190,0.5920,27,9.5049,3.4101


- **Letting "says no or goes quiet" depend on the customer is what matters.** One figure per type
  of work misses in 3 groups; one per customer scores better (0.5705 against 0.5717) and misses in 1.
- **Learning reply speed for smaller groups doesn't help.** By sending speed it ties on log loss, by
  customer source it's slightly worse, and both miss in more groups. So reply speed is learned per
  type of work.
- **This approach beats the usual alternative** of adding "days waiting" as one more fact (van
  Houwelingen 2007), and it keeps "days without a reply" as its own reason on the list.
- **Days waiting on its own is badly off for many groups** (27 misses), which matters for notebook 05.

So the model is the one in `config.toml`: the three facts, a "says no or goes quiet" chance for each
customer, and reply speeds learned per type of work.